# Notebook 22: Graph Posterior Final Adjudicator

Offline graph-posterior final adjudication over Notebook `13` traces.

This notebook keeps Notebook `13` unchanged as the evidence-acquisition system. It reconstructs each final visible evidence state, scores every pathology with a train-derived signed evidence graph from Notebook `16`, and tests whether a conservative graph critic can improve final top-1 diagnosis without API calls.

Primary method: **Conservative Graph Critic v1**.


In [10]:
from __future__ import annotations

import ast
import json
import math
import re
import time
from collections import Counter, OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
NOTEBOOK13_49_RUN = PROJECT_ROOT / "artifacts" / "sequential_hybrid_mlp_feedback" / "selected_stop_live_confirmation_49case_v1"
NOTEBOOK13_24_RUN = PROJECT_ROOT / "artifacts" / "sequential_hybrid_mlp_feedback" / "selected_stop_live_confirmation_24case_v1"
GRAPH_STATS_ROOT = PROJECT_ROOT / "artifacts" / "graph_algorithmic_ledger" / "medkgi_style_offline_notebook13_49case_v1"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "graph_algorithmic_ledger" / "graph_posterior_final_adjudicator_49case_v1"
FIGURE_DIR = ARTIFACT_ROOT / "figures"

for path in [ARTIFACT_ROOT, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RUN_NAME = "graph_posterior_final_adjudicator_49case_v1"
PRIMARY_POLICY_NAME = "conservative_graph_critic_v1_clip3_margin1"
PRIMARY_CLIP_VALUE = 3.0
PRIMARY_GRAPH_MARGIN_MIN = 1.0
PRIMARY_REQUIRE_REFERENCE_NEGATIVE = True
PRIMARY_REQUIRE_GRAPH_POSITIVE = True

RUNS = {
    "notebook13_49case": NOTEBOOK13_49_RUN,
    "notebook13_24case": NOTEBOOK13_24_RUN,
}

print("Project root:", PROJECT_ROOT)
print("Notebook 13 49-case:", NOTEBOOK13_49_RUN)
print("Notebook 13 24-case:", NOTEBOOK13_24_RUN)
print("Graph stats:", GRAPH_STATS_ROOT)
print("Artifact root:", ARTIFACT_ROOT)


Project root: /Users/alfred/Documents/baseline_model
Notebook 13 49-case: /Users/alfred/Documents/baseline_model/artifacts/sequential_hybrid_mlp_feedback/selected_stop_live_confirmation_49case_v1
Notebook 13 24-case: /Users/alfred/Documents/baseline_model/artifacts/sequential_hybrid_mlp_feedback/selected_stop_live_confirmation_24case_v1
Graph stats: /Users/alfred/Documents/baseline_model/artifacts/graph_algorithmic_ledger/medkgi_style_offline_notebook13_49case_v1
Artifact root: /Users/alfred/Documents/baseline_model/artifacts/graph_algorithmic_ledger/graph_posterior_final_adjudicator_49case_v1


## 1. Utility Functions

In [11]:
def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path: Path, payload: Any) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
        handle.write("\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def parse_list(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(x) for x in value]
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return []


def extract_evidence_roots(value: Any) -> list[str]:
    if isinstance(value, list):
        text = " ".join(map(str, value))
    else:
        text = "" if pd.isna(value) else str(value)
    return re.findall(r"E_\d+", text)


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def softmax_from_scores(scores: dict[str, float]) -> dict[str, float]:
    if not scores:
        return {}
    labels = list(scores)
    arr = np.array([scores[label] for label in labels], dtype=float)
    arr = arr - np.max(arr)
    exp = np.exp(arr)
    denom = float(exp.sum())
    if denom <= 0:
        return {label: 1.0 / len(labels) for label in labels}
    return {label: float(val / denom) for label, val in zip(labels, exp)}


def macro_f1(y_true: list[str], y_pred: list[str]) -> float:
    labels = sorted(set(y_true) | set(y_pred))
    scores = []
    for label in labels:
        tp = sum(t == label and p == label for t, p in zip(y_true, y_pred))
        fp = sum(t != label and p == label for t, p in zip(y_true, y_pred))
        fn = sum(t == label and p != label for t, p in zip(y_true, y_pred))
        if tp == 0 and fp == 0 and fn == 0:
            continue
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        scores.append(f1)
    return float(np.mean(scores)) if scores else 0.0


def summarize_predictions(df: pd.DataFrame, pred_col: str, ranked_col: str | None = None) -> dict[str, Any]:
    y_true = df["true_pathology"].astype(str).tolist()
    y_pred = df[pred_col].astype(str).tolist()
    correct = [t == p for t, p in zip(y_true, y_pred)]
    if ranked_col is None:
        top3 = correct
        top5 = correct
    else:
        ranked = df[ranked_col].tolist()
        top3 = [t in (lst[:3] if isinstance(lst, list) else []) for t, lst in zip(y_true, ranked)]
        top5 = [t in (lst[:5] if isinstance(lst, list) else []) for t, lst in zip(y_true, ranked)]
    return {
        "num_cases": int(len(df)),
        "correct_count": int(sum(correct)),
        "accuracy": float(np.mean(correct)) if correct else 0.0,
        "top3_accuracy": float(np.mean(top3)) if top3 else 0.0,
        "top5_accuracy": float(np.mean(top5)) if top5 else 0.0,
        "macro_f1": macro_f1(y_true, y_pred),
        "mean_requests": float(df["num_requests"].mean()) if "num_requests" in df else None,
        "median_requests": float(df["num_requests"].median()) if "num_requests" in df else None,
    }


## 2. Load Notebook 13 And Graph Artifacts

In [12]:
required_files = [
    NOTEBOOK13_49_RUN / "predictions.csv",
    NOTEBOOK13_49_RUN / "traces.jsonl",
    NOTEBOOK13_49_RUN / "metrics.json",
    NOTEBOOK13_24_RUN / "predictions.csv",
    NOTEBOOK13_24_RUN / "traces.jsonl",
    GRAPH_STATS_ROOT / "global_evidence_graph_edges.csv",
    GRAPH_STATS_ROOT / "root_outcome_statistics.csv",
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required input files:\n" + "\n".join(missing))

run_inputs: dict[str, dict[str, Any]] = {}
for run_name, run_root in RUNS.items():
    predictions = pd.read_csv(run_root / "predictions.csv")
    traces = read_jsonl(run_root / "traces.jsonl")
    metrics_path = run_root / "metrics.json"
    metrics = read_json(metrics_path) if metrics_path.exists() else {}
    run_inputs[run_name] = {
        "root": run_root,
        "predictions": predictions,
        "traces": traces,
        "metrics": metrics,
    }
    print(run_name, predictions.shape, "traces", len(traces))

graph_edges = pd.read_csv(GRAPH_STATS_ROOT / "global_evidence_graph_edges.csv")
root_stats = pd.read_csv(GRAPH_STATS_ROOT / "root_outcome_statistics.csv")
pathologies = sorted(graph_edges["pathology"].dropna().astype(str).unique().tolist())
edge_lookup: dict[tuple[str, str, str], tuple[float, float]] = {}
for row in graph_edges.itertuples(index=False):
    edge_lookup[(str(row.root_evidence_id), str(row.outcome_state), str(row.pathology))] = (
        float(row.log_odds_support),
        float(row.global_mi_norm),
    )

print("Graph edges:", graph_edges.shape)
print("Root stats:", root_stats.shape)
print("Pathologies:", len(pathologies))


notebook13_49case (49, 51) traces 49
notebook13_24case (24, 51) traces 24
Graph edges: (44443, 10)
Root stats: (223, 8)
Pathologies: 49


## 3. Reconstruct Final Evidence States

In [13]:
def reconstruct_final_evidence(predictions: pd.DataFrame, traces: list[dict[str, Any]]) -> tuple[pd.DataFrame, dict[str, list[dict[str, Any]]]]:
    pred_by_case = predictions.set_index("case_id", drop=False)
    evidence_rows: list[dict[str, Any]] = []
    evidence_by_case: dict[str, list[dict[str, Any]]] = {}

    for trace_obj in traces:
        case_id = str(trace_obj["case_id"])
        if case_id not in pred_by_case.index:
            raise KeyError(f"Trace case {case_id} missing from predictions")
        pred_row = pred_by_case.loc[case_id]
        ordered: OrderedDict[str, dict[str, Any]] = OrderedDict()

        for root_id in extract_evidence_roots(pred_row.get("initial_evidence")):
            ordered[root_id] = {
                "case_id": case_id,
                "root_evidence_id": root_id,
                "outcome_state": "present",
                "source": "initial_evidence",
                "turn_index": 0,
                "question_en": "",
                "summary": f"{root_id} -> present [initial]",
            }

        for turn in trace_obj.get("trace", []):
            reveal = turn.get("reveal_payload") or {}
            root_id = reveal.get("root_evidence_id")
            if not root_id:
                continue
            status = "present" if reveal.get("status") == "present" else "absent"
            # Reassigning an existing key updates the value while preserving deterministic uniqueness.
            ordered[str(root_id)] = {
                "case_id": case_id,
                "root_evidence_id": str(root_id),
                "outcome_state": status,
                "source": "request",
                "turn_index": int(turn.get("turn_index", 0) or 0),
                "question_en": reveal.get("question_en", ""),
                "summary": reveal.get("summary", ""),
            }

        rows = list(ordered.values())
        evidence_by_case[case_id] = rows
        evidence_rows.extend(rows)

    evidence_df = pd.DataFrame(evidence_rows)
    return evidence_df, evidence_by_case


all_evidence_frames = []
evidence_lookup_by_run: dict[str, dict[str, list[dict[str, Any]]]] = {}
for run_name, payload in run_inputs.items():
    evidence_df, evidence_by_case = reconstruct_final_evidence(payload["predictions"], payload["traces"])
    evidence_df.insert(0, "run_scope", run_name)
    all_evidence_frames.append(evidence_df)
    evidence_lookup_by_run[run_name] = evidence_by_case
    print(run_name, "evidence rows", len(evidence_df), "cases", len(evidence_by_case))

final_evidence_states = pd.concat(all_evidence_frames, ignore_index=True)
print(final_evidence_states.groupby(["run_scope", "outcome_state"]).size())


notebook13_49case evidence rows 372 cases 49
notebook13_24case evidence rows 182 cases 24
run_scope          outcome_state
notebook13_24case  absent           123
                   present           59
notebook13_49case  absent           246
                   present          126
dtype: int64


## 4. Graph Posterior And Conservative Critic

In [14]:
@dataclass(frozen=True)
class GraphPolicyConfig:
    policy_name: str
    clip_value: float
    margin_min: float
    require_reference_negative: bool = True
    require_graph_positive: bool = True
    diagnostic_only: bool = False


PRIMARY_POLICY = GraphPolicyConfig(
    policy_name=PRIMARY_POLICY_NAME,
    clip_value=PRIMARY_CLIP_VALUE,
    margin_min=PRIMARY_GRAPH_MARGIN_MIN,
    require_reference_negative=PRIMARY_REQUIRE_REFERENCE_NEGATIVE,
    require_graph_positive=PRIMARY_REQUIRE_GRAPH_POSITIVE,
    diagnostic_only=False,
)


def compute_graph_state(evidence_rows: list[dict[str, Any]], clip_value: float = 3.0) -> dict[str, Any]:
    net = {pathology: 0.0 for pathology in pathologies}
    positive = {pathology: 0.0 for pathology in pathologies}
    contradiction = {pathology: 0.0 for pathology in pathologies}
    used_edges = 0
    missing_edges = 0

    for evidence in evidence_rows:
        root_id = str(evidence["root_evidence_id"])
        outcome_state = str(evidence["outcome_state"])
        for pathology in pathologies:
            edge = edge_lookup.get((root_id, outcome_state, pathology))
            if edge is None:
                missing_edges += 1
                continue
            raw_weight, _global_mi_norm = edge
            weight = max(-clip_value, min(clip_value, raw_weight))
            net[pathology] += weight
            if weight >= 0:
                positive[pathology] += weight
            else:
                contradiction[pathology] += -weight
            used_edges += 1

    ranked = sorted(pathologies, key=lambda label: net[label], reverse=True)
    posterior = softmax_from_scores(net)
    top1 = ranked[0]
    top2 = ranked[1] if len(ranked) > 1 else ranked[0]
    return {
        "net": net,
        "positive": positive,
        "contradiction": contradiction,
        "posterior": posterior,
        "ranked": ranked,
        "top1": top1,
        "top2": top2,
        "margin": float(net[top1] - net[top2]),
        "used_edges": int(used_edges),
        "missing_edges": int(missing_edges),
    }


def reference_ranked(row: pd.Series) -> list[str]:
    ranked = parse_list(row.get("ranked_differential"))
    pred = str(row.get("predicted_pathology", ""))
    if pred and pred not in ranked:
        ranked = [pred] + ranked
    return ranked


def adjudicate_case(row: pd.Series, graph_state: dict[str, Any], config: GraphPolicyConfig) -> dict[str, Any]:
    reference_pred = str(row["predicted_pathology"])
    graph_top1 = str(graph_state["top1"])
    graph_scores = graph_state["net"]
    graph_ranked = graph_state["ranked"]
    reference_score = float(graph_scores.get(reference_pred, 0.0))
    graph_top_score = float(graph_scores.get(graph_top1, 0.0))
    margin = float(graph_state["margin"])
    should_override = graph_top1 != reference_pred and margin >= config.margin_min
    if config.require_reference_negative:
        should_override = should_override and reference_score < 0.0
    if config.require_graph_positive:
        should_override = should_override and graph_top_score > 0.0

    if should_override:
        predicted = graph_top1
        ranked = graph_ranked
        source = "graph_override"
    else:
        predicted = reference_pred
        ranked = reference_ranked(row)
        source = "notebook13_reference"

    return {
        "predicted_pathology": predicted,
        "ranked_differential": ranked,
        "decision_source": source,
        "override_fired": bool(should_override),
        "graph_top1": graph_top1,
        "graph_top2": graph_state["top2"],
        "graph_margin": margin,
        "graph_top1_score": graph_top_score,
        "reference_graph_score": reference_score,
        "graph_top1_posterior": float(graph_state["posterior"].get(graph_top1, 0.0)),
        "reference_graph_posterior": float(graph_state["posterior"].get(reference_pred, 0.0)),
    }


def build_graph_feature_tables(run_name: str, clip_value: float) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, dict[str, Any]]]:
    predictions = run_inputs[run_name]["predictions"]
    evidence_by_case = evidence_lookup_by_run[run_name]
    graph_states: dict[str, dict[str, Any]] = {}
    feature_rows: list[dict[str, Any]] = []
    case_rows: list[dict[str, Any]] = []

    for row in predictions.itertuples(index=False):
        case_id = str(row.case_id)
        row_series = predictions.loc[predictions["case_id"] == case_id].iloc[0]
        evidence_rows = evidence_by_case[case_id]
        graph_state = compute_graph_state(evidence_rows, clip_value=clip_value)
        graph_states[case_id] = graph_state
        true_pathology = str(row.true_pathology)
        reference_pred = str(row.predicted_pathology)

        for rank, pathology in enumerate(graph_state["ranked"], start=1):
            feature_rows.append({
                "run_scope": run_name,
                "case_id": case_id,
                "true_pathology": true_pathology,
                "notebook13_predicted_pathology": reference_pred,
                "disease": pathology,
                "graph_rank": rank,
                "graph_net_support": float(graph_state["net"][pathology]),
                "graph_positive_support": float(graph_state["positive"][pathology]),
                "graph_contradiction": float(graph_state["contradiction"][pathology]),
                "graph_posterior": float(graph_state["posterior"].get(pathology, 0.0)),
                "is_true_pathology": pathology == true_pathology,
                "is_notebook13_prediction": pathology == reference_pred,
                "clip_value": clip_value,
            })

        graph_top1 = graph_state["top1"]
        true_rank = graph_state["ranked"].index(true_pathology) + 1 if true_pathology in graph_state["ranked"] else 999
        case_rows.append({
            "run_scope": run_name,
            "case_id": case_id,
            "true_pathology": true_pathology,
            "notebook13_predicted_pathology": reference_pred,
            "notebook13_correct": bool(row.correct),
            "num_requests": int(row.num_requests),
            "stop_reason": str(row.stop_reason),
            "visible_evidence_roots": len(evidence_rows),
            "graph_top1": graph_top1,
            "graph_top2": graph_state["top2"],
            "graph_only_correct": graph_top1 == true_pathology,
            "graph_true_rank": true_rank,
            "graph_margin": float(graph_state["margin"]),
            "graph_top1_score": float(graph_state["net"][graph_top1]),
            "graph_true_score": float(graph_state["net"].get(true_pathology, 0.0)),
            "graph_reference_score": float(graph_state["net"].get(reference_pred, 0.0)),
            "graph_top1_posterior": float(graph_state["posterior"].get(graph_top1, 0.0)),
            "graph_reference_posterior": float(graph_state["posterior"].get(reference_pred, 0.0)),
            "graph_used_edges": graph_state["used_edges"],
            "graph_missing_edges": graph_state["missing_edges"],
            "notebook13_ranked_differential": reference_ranked(row_series),
            "graph_ranked_differential": graph_state["ranked"],
        })

    return pd.DataFrame(feature_rows), pd.DataFrame(case_rows), graph_states


feature_frames = []
case_feature_frames = []
graph_states_by_run: dict[tuple[str, float], dict[str, dict[str, Any]]] = {}
for run_name in RUNS:
    feature_df, case_df, graph_states = build_graph_feature_tables(run_name, clip_value=PRIMARY_CLIP_VALUE)
    feature_frames.append(feature_df)
    case_feature_frames.append(case_df)
    graph_states_by_run[(run_name, PRIMARY_CLIP_VALUE)] = graph_states

graph_final_state_features = pd.concat(feature_frames, ignore_index=True)
case_graph_features = pd.concat(case_feature_frames, ignore_index=True)
graph_final_state_features.to_csv(ARTIFACT_ROOT / "graph_final_state_features.csv", index=False)
case_graph_features.to_csv(ARTIFACT_ROOT / "case_graph_final_state_features.csv", index=False)

print("Graph final state features:", graph_final_state_features.shape)
print("Case graph features:", case_graph_features.shape)
case_graph_features.groupby("run_scope")[["notebook13_correct", "graph_only_correct"]].mean()


Graph final state features: (3577, 13)
Case graph features: (73, 22)


,notebook13_correct,graph_only_correct
run_scope,,
notebook13_24case,0.916667,0.958333
notebook13_49case,0.877551,0.897959


## 5. Policy Variant Evaluation

In [15]:
def evaluate_policy(run_name: str, config: GraphPolicyConfig) -> tuple[dict[str, Any], pd.DataFrame]:
    predictions = run_inputs[run_name]["predictions"].copy()
    evidence_by_case = evidence_lookup_by_run[run_name]
    rows: list[dict[str, Any]] = []

    for _, row in predictions.iterrows():
        case_id = str(row["case_id"])
        graph_state = compute_graph_state(evidence_by_case[case_id], clip_value=config.clip_value)
        decision = adjudicate_case(row, graph_state, config)
        graph_ranked = graph_state["ranked"]
        graph_only_pred = graph_ranked[0]
        selected_pred = decision["predicted_pathology"]
        selected_ranked = decision["ranked_differential"]
        true_pathology = str(row["true_pathology"])
        reference_pred = str(row["predicted_pathology"])

        rows.append({
            "run_scope": run_name,
            "policy_name": config.policy_name,
            "diagnostic_only": bool(config.diagnostic_only),
            "case_id": case_id,
            "true_pathology": true_pathology,
            "notebook13_predicted_pathology": reference_pred,
            "graph_only_predicted_pathology": graph_only_pred,
            "graph_adjudicator_predicted_pathology": selected_pred,
            "notebook13_correct": bool(row["correct"]),
            "graph_only_correct": graph_only_pred == true_pathology,
            "graph_adjudicator_correct": selected_pred == true_pathology,
            "graph_adjudicator_top3_correct": true_pathology in selected_ranked[:3],
            "graph_adjudicator_top5_correct": true_pathology in selected_ranked[:5],
            "graph_only_top3_correct": true_pathology in graph_ranked[:3],
            "graph_only_top5_correct": true_pathology in graph_ranked[:5],
            "num_requests": int(row["num_requests"]),
            "stop_reason": str(row["stop_reason"]),
            "override_fired": decision["override_fired"],
            "decision_source": decision["decision_source"],
            "graph_top1": decision["graph_top1"],
            "graph_top2": decision["graph_top2"],
            "graph_margin": decision["graph_margin"],
            "graph_top1_score": decision["graph_top1_score"],
            "reference_graph_score": decision["reference_graph_score"],
            "graph_top1_posterior": decision["graph_top1_posterior"],
            "reference_graph_posterior": decision["reference_graph_posterior"],
            "clip_value": config.clip_value,
            "margin_min": config.margin_min,
            "require_reference_negative": config.require_reference_negative,
            "require_graph_positive": config.require_graph_positive,
            "regression_vs_notebook13": bool(row["correct"]) and selected_pred != true_pathology,
            "improvement_vs_notebook13": (not bool(row["correct"])) and selected_pred == true_pathology,
            "win_loss_vs_notebook13": (
                "both_correct" if bool(row["correct"]) and selected_pred == true_pathology else
                "notebook13_only_correct" if bool(row["correct"]) and selected_pred != true_pathology else
                "graph_adjudicator_only_correct" if (not bool(row["correct"])) and selected_pred == true_pathology else
                "both_wrong"
            ),
            "selected_ranked_differential": selected_ranked,
            "graph_ranked_differential": graph_ranked,
            "notebook13_ranked_differential": reference_ranked(row),
        })

    case_results = pd.DataFrame(rows)
    metrics = summarize_predictions(
        case_results.rename(columns={"graph_adjudicator_predicted_pathology": "selected_pred"}),
        "selected_pred",
        ranked_col="selected_ranked_differential",
    )
    graph_only_metrics = summarize_predictions(
        case_results.rename(columns={"graph_only_predicted_pathology": "graph_only_pred"}),
        "graph_only_pred",
        ranked_col="graph_ranked_differential",
    )
    reference_metrics = summarize_predictions(
        predictions.assign(reference_ranked=predictions.apply(reference_ranked, axis=1)),
        "predicted_pathology",
        ranked_col="reference_ranked",
    )

    summary = {
        "run_scope": run_name,
        "policy_name": config.policy_name,
        "diagnostic_only": bool(config.diagnostic_only),
        "num_cases": metrics["num_cases"],
        "accuracy": metrics["accuracy"],
        "correct_count": metrics["correct_count"],
        "top3_accuracy": metrics["top3_accuracy"],
        "top5_accuracy": metrics["top5_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "mean_requests": metrics["mean_requests"],
        "median_requests": metrics["median_requests"],
        "notebook13_accuracy": reference_metrics["accuracy"],
        "notebook13_correct_count": reference_metrics["correct_count"],
        "notebook13_top3_accuracy": reference_metrics["top3_accuracy"],
        "notebook13_top5_accuracy": reference_metrics["top5_accuracy"],
        "graph_only_accuracy": graph_only_metrics["accuracy"],
        "graph_only_correct_count": graph_only_metrics["correct_count"],
        "graph_only_top3_accuracy": graph_only_metrics["top3_accuracy"],
        "graph_only_top5_accuracy": graph_only_metrics["top5_accuracy"],
        "changed_predictions": int((case_results["graph_adjudicator_predicted_pathology"] != case_results["notebook13_predicted_pathology"]).sum()),
        "overrides_fired": int(case_results["override_fired"].sum()),
        "improvements_vs_notebook13": int(case_results["improvement_vs_notebook13"].sum()),
        "regressions_vs_notebook13": int(case_results["regression_vs_notebook13"].sum()),
        "clip_value": config.clip_value,
        "margin_min": config.margin_min,
        "require_reference_negative": config.require_reference_negative,
        "require_graph_positive": config.require_graph_positive,
    }
    return summary, case_results


policy_configs = [PRIMARY_POLICY]
for clip_value in [1.0, 2.0, 3.0, 5.0, 10.0]:
    for margin_min in [0.0, 0.5, 1.0, 1.5, 2.0]:
        name = f"diagnostic_clip{clip_value:g}_margin{margin_min:g}"
        if clip_value == PRIMARY_CLIP_VALUE and margin_min == PRIMARY_GRAPH_MARGIN_MIN:
            name = f"{name}_same_thresholds"
        policy_configs.append(GraphPolicyConfig(
            policy_name=name,
            clip_value=clip_value,
            margin_min=margin_min,
            require_reference_negative=True,
            require_graph_positive=True,
            diagnostic_only=True,
        ))

summary_rows: list[dict[str, Any]] = []
case_result_frames: list[pd.DataFrame] = []
for run_name in RUNS:
    for config in policy_configs:
        summary, case_results = evaluate_policy(run_name, config)
        summary_rows.append(summary)
        case_result_frames.append(case_results)

policy_summary = pd.DataFrame(summary_rows)
case_level_results = pd.concat(case_result_frames, ignore_index=True)
policy_summary.to_csv(ARTIFACT_ROOT / "graph_adjudicator_policy_summary.csv", index=False)
case_level_results.to_csv(ARTIFACT_ROOT / "case_level_graph_adjudicator_results.csv", index=False)

primary_summary = policy_summary[(policy_summary["run_scope"] == "notebook13_49case") & (policy_summary["policy_name"] == PRIMARY_POLICY_NAME)].iloc[0]
print(primary_summary.to_string())
policy_summary.sort_values(["run_scope", "diagnostic_only", "accuracy", "regressions_vs_notebook13"], ascending=[True, True, False, True]).head(12)


run_scope                                              notebook13_49case
policy_name                   conservative_graph_critic_v1_clip3_margin1
diagnostic_only                                                    False
num_cases                                                             49
accuracy                                                        0.897959
correct_count                                                         44
top3_accuracy                                                   0.938776
top5_accuracy                                                   0.938776
macro_f1                                                        0.867347
mean_requests                                                   6.591837
median_requests                                                      5.0
notebook13_accuracy                                             0.877551
notebook13_correct_count                                              43
notebook13_top3_accuracy                           

,run_scope,policy_name,diagnostic_only,num_cases,accuracy,correct_count,top3_accuracy,top5_accuracy,macro_f1,mean_requests,...,graph_only_top3_accuracy,graph_only_top5_accuracy,changed_predictions,overrides_fired,improvements_vs_notebook13,regressions_vs_notebook13,clip_value,margin_min,require_reference_negative,require_graph_positive
26,notebook13_24case,conservative_graph_critic_v1_clip3_margin1,False,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,3.0,1.0,True,True
32,notebook13_24case,diagnostic_clip2_margin0,True,24,0.958333,23,0.958333,0.958333,0.944444,6.583333,...,0.958333,0.958333,2,2,1,0,2.0,0.0,True,True
33,notebook13_24case,diagnostic_clip2_margin0.5,True,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,2.0,0.5,True,True
34,notebook13_24case,diagnostic_clip2_margin1,True,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,2.0,1.0,True,True
37,notebook13_24case,diagnostic_clip3_margin0,True,24,0.958333,23,0.958333,0.958333,0.944444,6.583333,...,0.958333,0.958333,2,2,1,0,3.0,0.0,True,True
38,notebook13_24case,diagnostic_clip3_margin0.5,True,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,3.0,0.5,True,True
39,notebook13_24case,diagnostic_clip3_margin1_same_thresholds,True,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,3.0,1.0,True,True
40,notebook13_24case,diagnostic_clip3_margin1.5,True,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,3.0,1.5,True,True
41,notebook13_24case,diagnostic_clip3_margin2,True,24,0.958333,23,0.958333,0.958333,0.920000,6.583333,...,0.958333,0.958333,1,1,1,0,3.0,2.0,True,True
42,notebook13_24case,diagnostic_clip5_margin0,True,24,0.958333,23,0.958333,0.958333,0.944444,6.583333,...,0.958333,0.958333,2,2,1,0,5.0,0.0,True,True


## 6. Paired Error Analysis

In [16]:
primary_cases = case_level_results[
    (case_level_results["run_scope"] == "notebook13_49case")
    & (case_level_results["policy_name"] == PRIMARY_POLICY_NAME)
].copy()
paired_cols = [
    "case_id", "true_pathology", "notebook13_predicted_pathology", "graph_only_predicted_pathology",
    "graph_adjudicator_predicted_pathology", "notebook13_correct", "graph_only_correct",
    "graph_adjudicator_correct", "win_loss_vs_notebook13", "override_fired", "decision_source",
    "num_requests", "stop_reason", "graph_margin", "graph_top1_score", "reference_graph_score",
    "graph_top1_posterior", "reference_graph_posterior",
]
paired = primary_cases[paired_cols].sort_values(["win_loss_vs_notebook13", "case_id"])
paired.to_csv(ARTIFACT_ROOT / "paired_notebook13_vs_graph_adjudicator.csv", index=False)

changed_cases = primary_cases[primary_cases["notebook13_predicted_pathology"] != primary_cases["graph_adjudicator_predicted_pathology"]].copy()
notebook13_errors = primary_cases[~primary_cases["notebook13_correct"]].copy()
print("Changed predictions:", len(changed_cases))
print(changed_cases[paired_cols].to_string(index=False))
print("\nNotebook 13 errors under graph critic:")
print(notebook13_errors[paired_cols].to_string(index=False))

hard_case_ids = sorted(set(changed_cases["case_id"].tolist()) | set(notebook13_errors["case_id"].tolist()))
hard_case_audits: dict[str, Any] = {}
feature_by_case = graph_final_state_features[
    (graph_final_state_features["run_scope"] == "notebook13_49case")
    & (graph_final_state_features["clip_value"] == PRIMARY_CLIP_VALUE)
]
for case_id in hard_case_ids:
    case_row = primary_cases[primary_cases["case_id"] == case_id].iloc[0]
    disease_rows = feature_by_case[feature_by_case["case_id"] == case_id].sort_values("graph_rank").head(10)
    evidence_rows = evidence_lookup_by_run["notebook13_49case"][case_id]
    hard_case_audits[case_id] = {
        "true_pathology": case_row["true_pathology"],
        "notebook13_predicted_pathology": case_row["notebook13_predicted_pathology"],
        "graph_only_predicted_pathology": case_row["graph_only_predicted_pathology"],
        "graph_adjudicator_predicted_pathology": case_row["graph_adjudicator_predicted_pathology"],
        "notebook13_correct": bool(case_row["notebook13_correct"]),
        "graph_adjudicator_correct": bool(case_row["graph_adjudicator_correct"]),
        "override_fired": bool(case_row["override_fired"]),
        "num_requests": int(case_row["num_requests"]),
        "stop_reason": case_row["stop_reason"],
        "graph_margin": float(case_row["graph_margin"]),
        "reference_graph_score": float(case_row["reference_graph_score"]),
        "graph_top1_score": float(case_row["graph_top1_score"]),
        "top_graph_diagnoses": disease_rows[[
            "disease", "graph_rank", "graph_net_support", "graph_positive_support", "graph_contradiction", "graph_posterior",
            "is_true_pathology", "is_notebook13_prediction",
        ]].to_dict(orient="records"),
        "final_visible_evidence": evidence_rows,
    }
write_json(ARTIFACT_ROOT / "hard_case_graph_adjudicator_audits.json", hard_case_audits)

win_loss_counts = primary_cases["win_loss_vs_notebook13"].value_counts().to_dict()
win_loss_counts


Changed predictions: 1
   case_id true_pathology notebook13_predicted_pathology graph_only_predicted_pathology graph_adjudicator_predicted_pathology  notebook13_correct  graph_only_correct  graph_adjudicator_correct         win_loss_vs_notebook13  override_fired decision_source  num_requests stop_reason  graph_margin  graph_top1_score  reference_graph_score  graph_top1_posterior  reference_graph_posterior
test:81691          Croup                         Anemia                          Croup                                 Croup               False                True                       True graph_adjudicator_only_correct            True  graph_override            19  agent_stop      2.072659          1.177246              -2.359106              0.494811                   0.014409

Notebook 13 errors under graph critic:
    case_id                      true_pathology notebook13_predicted_pathology graph_only_predicted_pathology graph_adjudicator_predicted_pathology  notebook13_corre

{'both_correct': 43, 'both_wrong': 5, 'graph_adjudicator_only_correct': 1}

## 7. Figures

In [17]:
primary_plot = policy_summary[
    (policy_summary["run_scope"] == "notebook13_49case")
    & (policy_summary["policy_name"].isin([PRIMARY_POLICY_NAME]))
].copy()
reference_acc = float(primary_summary["notebook13_accuracy"])
graph_only_acc = float(primary_summary["graph_only_accuracy"])
critic_acc = float(primary_summary["accuracy"])

fig, ax = plt.subplots(figsize=(7, 4))
labels = ["Notebook 13", "Graph-only", "Conservative critic"]
values = [reference_acc, graph_only_acc, critic_acc]
colors = ["#4C78A8", "#F58518", "#54A24B"]
ax.bar(labels, values, color=colors)
ax.set_ylim(0.80, 0.93)
ax.set_ylabel("Accuracy")
ax.set_title("Notebook 22 final-head accuracy on Notebook 13 49-case traces")
for i, value in enumerate(values):
    ax.text(i, value + 0.004, f"{value:.3f}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "accuracy_comparison.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
counts = primary_cases["win_loss_vs_notebook13"].value_counts().reindex([
    "both_correct", "graph_adjudicator_only_correct", "notebook13_only_correct", "both_wrong"
]).fillna(0)
ax.bar(counts.index, counts.values, color=["#72B7B2", "#54A24B", "#E45756", "#B279A2"])
ax.set_ylabel("Cases")
ax.set_title("Paired outcome vs Notebook 13")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "paired_outcomes.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
error_rows = notebook13_errors.sort_values("graph_margin", ascending=False)
ax.scatter(error_rows["reference_graph_score"], error_rows["graph_top1_score"], s=70, color="#F58518")
for _, row in error_rows.iterrows():
    ax.annotate(row["true_pathology"].split(" /")[0][:18], (row["reference_graph_score"], row["graph_top1_score"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.axvline(0.0, color="#777777", linewidth=1, linestyle="--")
ax.axhline(0.0, color="#777777", linewidth=1, linestyle="--")
ax.set_xlabel("Graph score for Notebook 13 prediction")
ax.set_ylabel("Graph score for graph top-1")
ax.set_title("Notebook 13 errors under graph scoring")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "notebook13_error_graph_scores.png", dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
frontier = policy_summary[policy_summary["run_scope"] == "notebook13_49case"].copy()
ax.scatter(frontier["overrides_fired"], frontier["accuracy"], c=frontier["regressions_vs_notebook13"], cmap="coolwarm", s=55)
ax.scatter([int(primary_summary["overrides_fired"])], [float(primary_summary["accuracy"])], color="#54A24B", s=110, marker="*", label="selected")
ax.axhline(reference_acc, color="#4C78A8", linestyle="--", linewidth=1, label="Notebook 13")
ax.set_xlabel("Overrides fired")
ax.set_ylabel("Accuracy")
ax.set_title("Diagnostic sensitivity variants")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sensitivity_frontier.png", dpi=180)
plt.close(fig)

print("Figures saved to", FIGURE_DIR)


Figures saved to /Users/alfred/Documents/baseline_model/artifacts/graph_algorithmic_ledger/graph_posterior_final_adjudicator_49case_v1/figures


## 8. Final Summary And Artifact Contract

In [18]:
reference_49 = summarize_predictions(
    run_inputs["notebook13_49case"]["predictions"].assign(
        reference_ranked=run_inputs["notebook13_49case"]["predictions"].apply(reference_ranked, axis=1)
    ),
    "predicted_pathology",
    ranked_col="reference_ranked",
)
if reference_49["correct_count"] != 43:
    raise AssertionError(f"Expected Notebook 13 49-case correct_count 43, got {reference_49['correct_count']}")
if not math.isclose(reference_49["accuracy"], 43 / 49, rel_tol=0, abs_tol=1e-12):
    raise AssertionError(f"Unexpected Notebook 13 accuracy: {reference_49['accuracy']}")
if not math.isclose(reference_49["top3_accuracy"], 45 / 49, rel_tol=0, abs_tol=1e-12):
    raise AssertionError(f"Unexpected Notebook 13 top3: {reference_49['top3_accuracy']}")
if not math.isclose(reference_49["top5_accuracy"], 46 / 49, rel_tol=0, abs_tol=1e-12):
    raise AssertionError(f"Unexpected Notebook 13 top5: {reference_49['top5_accuracy']}")

primary_correct = int(primary_summary["correct_count"])
reference_correct = int(primary_summary["notebook13_correct_count"])
regressions = int(primary_summary["regressions_vs_notebook13"])
improvements = int(primary_summary["improvements_vs_notebook13"])
promotion_status = "offline_candidate_promoted" if primary_correct > reference_correct and regressions == 0 else "diagnostic_only_keep_notebook13"

selected_graph_adjudicator = {
    "policy_name": PRIMARY_POLICY_NAME,
    "status": promotion_status,
    "method": "Conservative final-state graph critic over Notebook 13 revealed evidence",
    "formula": "sum clipped train-derived log_odds_support(outcome -> disease) over visible final evidence",
    "clip_value": PRIMARY_CLIP_VALUE,
    "override_rule": {
        "graph_top1_differs_from_notebook13": True,
        "graph_margin_min": PRIMARY_GRAPH_MARGIN_MIN,
        "require_notebook13_graph_score_lt_0": PRIMARY_REQUIRE_REFERENCE_NEGATIVE,
        "require_graph_top1_score_gt_0": PRIMARY_REQUIRE_GRAPH_POSITIVE,
    },
    "inputs": {
        "notebook13_49_run": str(NOTEBOOK13_49_RUN),
        "notebook13_24_run": str(NOTEBOOK13_24_RUN),
        "graph_stats_root": str(GRAPH_STATS_ROOT),
    },
    "notebook13_reference": reference_49,
    "selected_policy_49case": primary_summary.to_dict(),
    "paired_counts": win_loss_counts,
    "promotion_rule": "Promote as offline candidate only if accuracy improves over Notebook 13 with zero regressions.",
    "external_validation_note": "This is an offline final-head adjudicator on saved Notebook 13 traces. A future live or held-out confirmation should keep Notebook 13 acquisition unchanged and validate the final critic separately.",
}
write_json(ARTIFACT_ROOT / "selected_graph_adjudicator.json", selected_graph_adjudicator)

resolved_run_config = {
    "run_name": RUN_NAME,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "offline_only": True,
    "api_calls_allowed": False,
    "primary_policy_name": PRIMARY_POLICY_NAME,
    "primary_clip_value": PRIMARY_CLIP_VALUE,
    "primary_graph_margin_min": PRIMARY_GRAPH_MARGIN_MIN,
    "artifact_root": str(ARTIFACT_ROOT),
    "inputs": selected_graph_adjudicator["inputs"],
    "required_artifacts": [
        "resolved_run_config.json",
        "graph_final_state_features.csv",
        "graph_adjudicator_policy_summary.csv",
        "case_level_graph_adjudicator_results.csv",
        "paired_notebook13_vs_graph_adjudicator.csv",
        "hard_case_graph_adjudicator_audits.json",
        "selected_graph_adjudicator.json",
    ],
}
write_json(ARTIFACT_ROOT / "resolved_run_config.json", resolved_run_config)

required_outputs = [ARTIFACT_ROOT / name for name in resolved_run_config["required_artifacts"]]
required_outputs += [
    FIGURE_DIR / "accuracy_comparison.png",
    FIGURE_DIR / "paired_outcomes.png",
    FIGURE_DIR / "notebook13_error_graph_scores.png",
    FIGURE_DIR / "sensitivity_frontier.png",
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise FileNotFoundError("Missing expected outputs:\n" + "\n".join(missing_outputs))

print(json.dumps({
    "notebook13_correct": reference_correct,
    "graph_critic_correct": primary_correct,
    "improvements_vs_notebook13": improvements,
    "regressions_vs_notebook13": regressions,
    "promotion_status": promotion_status,
}, indent=2))
print("Saved artifacts to", ARTIFACT_ROOT)


{
  "notebook13_correct": 43,
  "graph_critic_correct": 44,
  "improvements_vs_notebook13": 1,
  "regressions_vs_notebook13": 0,
  "promotion_status": "offline_candidate_promoted"
}
Saved artifacts to /Users/alfred/Documents/baseline_model/artifacts/graph_algorithmic_ledger/graph_posterior_final_adjudicator_49case_v1
